<!-- notebook-header -->
# Word Embeddings

**Modulo:** 05 - Dominios Aplicados / 05B - NLP  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Word2Vec, GloVe, similaridade semantica, analogias e representacoes distribuidas.


# 5B_2: Word Embeddings

## Visao Geral

Neste notebook, exploraremos representacoes densas de palavras (word embeddings),
desde a motivacao ate implementacoes de Word2Vec, GloVe e FastText.
Embeddings sao a base de TODA NLP moderna -- de BERT a GPT.

**Conteudo:**
1. Limitacoes de Representacoes Sparse
2. Word2Vec (Skip-gram e CBOW)
3. GloVe (Global Vectors)
4. FastText (Subword Embeddings)
5. Visualizacao e Avaliacao
6. Limitacoes e Contexto
7. Exercicios Praticos
8. Erros Comuns

**Pre-requisitos:** 5B_1 (NLP classico, BoW, TF-IDF)

**Dependencias:** numpy, matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
import math
print('Imports OK')

## 1. Limitacoes de Representacoes Sparse

### Analogia: Dicionario vs Enciclopedia

BoW/TF-IDF e como um dicionario: cada palavra e uma entrada SEPARADA.
"Rei" e "Rainha" sao paginas completamente diferentes, sem conexao.

Word embeddings sao como uma enciclopedia: "Rei" e "Rainha" estao no mesmo
capitulo (realeza), proximos um do outro. A POSICAO no livro carrega significado.

### Por que em ML representacoes sparse sao problematicas

1. **Alta dimensionalidade:** 100K palavras = 100K features (curse of dimensionality)
2. **Esparsidade:** 99.9% dos valores sao zero (ineficiente)
3. **Sem semantica:** cos("rei", "rainha") = 0 em BoW (sao features ortogonais)
4. **Sem generalizacao:** "bom" e "otimo" sao features completamente diferentes

### O que observar sobre a Hipotese Distribucional

"You shall know a word by the company it keeps" (Firth, 1957).
Palavras que aparecem em CONTEXTOS similares tem SIGNIFICADOS similares.
- "gato" e "cachorro" aparecem com "comida", "veterinario", "brinquedo"
- "gato" e "investimento" quase nunca aparecem juntos
- Portanto, embedding("gato") deve ser proximo de embedding("cachorro")

### O que concluir sobre o que Embeddings Capturam

Embeddings capturam RELACOES SEMANTICAS como operacoes ARITMETICAS:
- king - man + woman = queen
- Paris - France + Italy = Rome
- bigger - big + small = smaller

Isso emerge AUTOMATICAMENTE do treinamento, sem supervisao explicita!

### Conexao com outros notebooks sobre Representacao

Embeddings sao uma forma de reducao de dimensionalidade (3_2) que preserva
semantica. Assim como PCA projeta dados em dimensoes menores, embeddings
projetam palavras de 100K dims (BoW) para 300 dims (embedding denso).

In [ ]:

# Exemplo de relações capturadas por embeddings
# Em um espaço bem treinado, vetores satisfazem relações como:
# v(rei) - v(homem) + v(mulher) ≈ v(rainha)

# Simulação de embeddings
np.random.seed(42)
dim = 4

# Criar embeddings conceituais
embeddings = {
    'homem': np.array([0.8, 0.1, 0.2, 0.3]),
    'mulher': np.array([0.7, 0.2, 0.3, 0.4]),
    'rei': np.array([0.9, 0.5, 0.2, 0.1]),
    'rainha': np.array([0.85, 0.55, 0.3, 0.2]),
    'gato': np.array([0.3, 0.7, 0.1, 0.2]),
    'gato_feminino': np.array([0.25, 0.75, 0.2, 0.3])
}

# Verificar analogia: rei - homem + mulher ≈ rainha
analogy = embeddings['rei'] - embeddings['homem'] + embeddings['mulher']
print("Analogia: rei - homem + mulher =")
print(f"  Resultado: {analogy}")
print(f"  Rainha real: {embeddings['rainha']}")
print(f"  Distância: {np.linalg.norm(analogy - embeddings['rainha']):.3f}")


## 2. Word2Vec

### Analogia: Prever Vizinhos Revela Identidade

Imagine que voce so sabe QUEM anda com quem. Se joao anda com maria, pedro e
lucia, voce infere que joao e uma pessoa social. Se tiago anda com maria, pedro
e lucas, voce infere que tiago e SIMILAR a joao (mesmos amigos).

Word2Vec aprende embeddings prevendo vizinhos de palavras:
- **Skip-gram:** Dada uma palavra, prever as vizinhas
- **CBOW:** Dadas as vizinhas, prever a palavra central

### Por que em ML Skip-gram funciona melhor para palavras raras

Skip-gram: 1 input -> N outputs (palavras-contexto). Cada output e um exemplo.
CBOW: N inputs -> 1 output. So 1 exemplo por janela.

Para palavras raras, Skip-gram gera MAIS exemplos de treino, aprendendo melhor.
Para palavras frequentes, CBOW e mais rapido e funciona igual.

### O que observar sobre Negative Sampling

Treinar sobre TAREFA DO ALUNO o vocabulario e caro (softmax sobre 100K palavras).
Negative Sampling: para cada par positivo (word, context), sample K pares
negativos (word, random_word). Transforma o problema de classificacao em
K+1 classificacoes binarias -- muito mais rapido.

### O que concluir sobre Word2Vec como Revolução

Word2Vec (2013) foi a primeira demonstracao pratica de que embeddings densos
capturam semantica. Antes, NLP era dominado por features manuais (TF-IDF, n-grams).
Depois de Word2Vec, representacoes aprendidas se tornaram o padrao.

### Conexao com outros notebooks sobre Aprendizado de Representacoes

Word2Vec e aprendizado de representacoes (representation learning), conectando
com autoencoders (5C) e self-supervised learning (5A_5). A ideia central e a
mesma: aprender features uteis sem labels explicitas.

In [ ]:

# Implementação simplificada de Skip-gram
class SimpleWord2Vec:
    def __init__(self, vocab_size, embedding_dim=5, window_size=2):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.window_size = window_size
        
        # Embeddings aleatórios
        self.word_embeddings = np.random.randn(vocab_size, embedding_dim) * 0.01
        self.context_embeddings = np.random.randn(vocab_size, embedding_dim) * 0.01
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def train_pair(self, word_idx, context_idx, learning_rate=0.01):
        # Skip-gram: prever palavra contexto dado word
        word_vec = self.word_embeddings[word_idx]
        context_vec = self.context_embeddings[context_idx]
        
        # Score e predição
        score = np.dot(word_vec, context_vec)
        pred = self.sigmoid(score)
        
        # Gradient descent (simplificado)
        error = pred - 1  # Target é 1 para exemplos positivos
        
        # Atualizar embeddings
        self.word_embeddings[word_idx] -= learning_rate * error * context_vec
        self.context_embeddings[context_idx] -= learning_rate * error * word_vec
        
        return pred
    
    def get_embedding(self, word_idx):
        return self.word_embeddings[word_idx]

# Teste
vocab = {'gato': 0, 'dormiu': 1, 'árvore': 2, 'bonito': 3, 'cachorro': 4}
vocab_size = len(vocab)

w2v = SimpleWord2Vec(vocab_size, embedding_dim=3, window_size=2)

# Simular treinamento
for epoch in range(10):
    w2v.train_pair(vocab['gato'], vocab['dormiu'], learning_rate=0.05)
    w2v.train_pair(vocab['cachorro'], vocab['dormiu'], learning_rate=0.05)

print("Embeddings aprendidos (após 10 épocas):")
for word, idx in sorted(vocab.items(), key=lambda x: x[1]):
    embedding = w2v.get_embedding(idx)
    print(f"  {word:10}: {embedding}")


In [ ]:

# Calcular similaridade entre embeddings
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2 + 1e-8)

# Comparar similaridades
embeddings_dict = {
    'gato': w2v.get_embedding(vocab['gato']),
    'cachorro': w2v.get_embedding(vocab['cachorro']),
    'dormiu': w2v.get_embedding(vocab['dormiu']),
    'árvore': w2v.get_embedding(vocab['árvore'])
}

print("Similaridades (cosseno):")
word1 = 'gato'
for word2 in ['cachorro', 'dormiu', 'árvore']:
    sim = cosine_similarity(embeddings_dict[word1], embeddings_dict[word2])
    print(f"  {word1} vs {word2}: {sim:.3f}")


In [ ]:
# Visualizacao: aritmetica de embeddings
np.random.seed(42)

# Simular embeddings 2D para visualizacao
words = {
    'rei': np.array([1.0, 2.0]),
    'rainha': np.array([0.8, 2.3]),
    'homem': np.array([1.5, 0.5]),
    'mulher': np.array([1.3, 0.8]),
    'principe': np.array([1.2, 1.8]),
    'princesa': np.array([1.0, 2.1]),
    'paris': np.array([-1.5, 1.5]),
    'franca': np.array([-1.8, 0.5]),
    'roma': np.array([-0.5, 1.3]),
    'italia': np.array([-0.8, 0.3]),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Analogia: rei - homem + mulher = rainha
ax = axes[0]
for word, vec in words.items():
    if word in ['rei', 'rainha', 'homem', 'mulher', 'principe', 'princesa']:
        color = 'blue' if word in ['rei', 'homem', 'principe'] else 'red'
        ax.scatter(vec[0], vec[1], s=100, c=color, zorder=5, edgecolors='black')
        ax.annotate(word, (vec[0], vec[1]), fontsize=10, fontweight='bold',
                    xytext=(5, 5), textcoords='offset points')

# Vetor de genero
gender_vec = words['mulher'] - words['homem']
result = words['rei'] + gender_vec
ax.annotate('', xy=result, xytext=words['rei'],
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.scatter(result[0], result[1], s=150, c='green', marker='*', zorder=10,
          edgecolors='black', linewidth=1.5)
ax.annotate('rei-homem+mulher', (result[0], result[1]), fontsize=8, color='green',
            xytext=(5, -15), textcoords='offset points')

ax.set_title('Analogia: rei - homem + mulher = rainha', fontsize=11, fontweight='bold')
ax.set_xlabel('Dimensao 1')
ax.set_ylabel('Dimensao 2')
ax.legend(['Masculino', 'Feminino'], fontsize=9)
ax.grid(True, alpha=0.3)

# Analogia: paris - franca + italia = roma
ax = axes[1]
for word, vec in words.items():
    if word in ['paris', 'franca', 'roma', 'italia']:
        color = 'blue' if word in ['paris', 'roma'] else 'orange'
        ax.scatter(vec[0], vec[1], s=100, c=color, zorder=5, edgecolors='black')
        ax.annotate(word, (vec[0], vec[1]), fontsize=10, fontweight='bold',
                    xytext=(5, 5), textcoords='offset points')

country_vec = words['italia'] - words['franca']
result = words['paris'] + country_vec
ax.annotate('', xy=result, xytext=words['paris'],
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
ax.scatter(result[0], result[1], s=150, c='green', marker='*', zorder=10,
          edgecolors='black', linewidth=1.5)
ax.annotate('paris-franca+italia', (result[0], result[1]), fontsize=8, color='green',
            xytext=(5, -15), textcoords='offset points')

ax.set_title('Analogia: paris - franca + italia = roma', fontsize=11, fontweight='bold')
ax.set_xlabel('Dimensao 1')
ax.set_ylabel('Dimensao 2')
ax.legend(['Capitais', 'Paises'], fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/embedding_arithmetic.png', dpi=100, bbox_inches='tight')
plt.show()

print('Aritmetica de embeddings: operacoes vetoriais capturam relacoes semanticas')
print('Direcao "genero": mulher - homem = rainha - rei = princesa - principe')
print('Direcao "capital": paris - franca = roma - italia')

## 3. GloVe (Global Vectors)

### Analogia: Word2Vec le um Livro, GloVe le o Indice

Word2Vec olha CONTEXTOS LOCAIS (janela de 5-10 palavras).
GloVe olha a ESTATISTICA GLOBAL de co-ocorrencia do corpus inteiro.

E como a diferenca entre ler um livro pagina por pagina (Word2Vec)
vs ler o indice remissivo completo (GloVe).

### Por que em ML GloVe captura relacoes globais

GloVe otimiza: dot(w_i, w_j) = log(co-ocorrencia(i, j)).
Isso forca os vetores a codificarem a PROBABILIDADE de co-ocorrencia,
que reflete relacoes semanticas globais.

### O que observar sobre Word2Vec vs GloVe

| Aspecto | Word2Vec | GloVe |
|---------|----------|-------|
| Treino | Online (janela local) | Batch (matriz global) |
| Captura | Relacoes locais | Relacoes globais |
| Eficiencia | Mais rapido para corpus grande | Matriz de co-ocorrencia cara |
| Analogias | Bom | Ligeiramente melhor |
| Resultado pratico | Similares | Similares |

### O que concluir sobre a Equivalencia Pratica

Na pratica, Word2Vec e GloVe produzem embeddings de qualidade SIMILAR.
A escolha depende mais do framework disponivel que da qualidade.
Ambos sao superados por embeddings contextuais (BERT, 5B_4).

### Conexao com outros notebooks sobre Fatoracao de Matrizes

GloVe e essencialmente uma fatoracao de matrizes (SVD de co-ocorrencia),
conectando com 3_2 (PCA/SVD). A matriz de co-ocorrencia e analoga
a matriz de features, e os fatores sao os embeddings.

In [ ]:

# Construir matriz de co-ocorrência
class CooccurrenceMatrix:
    def __init__(self, window_size=2):
        self.window_size = window_size
        self.vocab = {}
        self.cooccurrence = defaultdict(lambda: defaultdict(int))
    
    def build(self, texts):
        # Criar vocabulário
        word_freq = Counter()
        for text in texts:
            words = text.lower().split()
            word_freq.update(words)
        
        self.vocab = {word: idx for idx, word in enumerate(word_freq.keys())}
        
        # Construir matriz de co-ocorrência
        for text in texts:
            words = text.lower().split()
            for i, word in enumerate(words):
                for j in range(max(0, i-self.window_size), 
                              min(len(words), i+self.window_size+1)):
                    if i != j:
                        w1_idx = self.vocab[word]
                        w2_idx = self.vocab[words[j]]
                        distance = abs(i - j)
                        weight = 1.0 / distance  # Palavras mais próximas têm peso maior
                        self.cooccurrence[w1_idx][w2_idx] += weight
    
    def get_matrix(self):
        size = len(self.vocab)
        matrix = np.zeros((size, size))
        for i in self.cooccurrence:
            for j in self.cooccurrence[i]:
                matrix[i][j] = self.cooccurrence[i][j]
        return matrix

# Teste
texts = [
    'gato dorme na árvore',
    'gato pula na árvore',
    'cachorro corre no parque'
]

cooc = CooccurrenceMatrix(window_size=2)
cooc.build(texts)
cooc_matrix = cooc.get_matrix()

print("Matriz de co-ocorrência:")
print(f"Vocabulário: {cooc.vocab}")
print(f"\nMatriz (primeiras 4x4):")
print(cooc_matrix[:4, :4])


## 4. FastText (Subword Embeddings)

### Analogia: Aprender Radicais em vez de Palavras Inteiras

Em portugues, se voce sabe que "corr-" significa "correr", voce entende
"correndo", "corrida", "corredor" mesmo sem ter visto essas palavras antes.

FastText faz o mesmo: decompoe palavras em subwords (n-gramas de caracteres)
e aprende embeddings dos pedacos. O embedding da palavra e a SOMA dos subwords.

### Por que em ML FastText resolve o problema de OOV

OOV = Out of Vocabulary (palavras nao vistas no treino).
- Word2Vec: "desempenhar" nao esta no vocabulario? Retorna ZERO.
- FastText: "desempenhar" = "<des" + "dese" + "esem" + ... -> soma dos subwords!

Isso e CRITICO para linguas com muita flexao (portugues, alemao, turco).

### O que observar sobre Eficiencia de FastText

FastText e mais lento para TREINAR (mais exemplos por palavra -- cada subword),
mas mais ROBUSTO (handle OOV, erros de digitacao, neologismos).
Na pratica, FastText e o melhor embedding estatico para a maioria dos idiomas.

### O que concluir sobre a Evolucao dos Embeddings

| Modelo | Ano | Inovacao | Limitacao Principal |
|--------|-----|----------|-------------------|
| Word2Vec | 2013 | Embeddings densos | OOV, polissemia |
| GloVe | 2014 | Co-ocorrencia global | OOV, polissemia |
| FastText | 2017 | Subwords | Polissemia |
| ELMo | 2018 | Contextual (LSTM) | Bidirecional limitado |
| BERT | 2019 | Contextual (Transformer) | Custo computacional |

### Conexao com outros notebooks sobre Tokenizacao

FastText com subwords e o precursor do BPE (Byte Pair Encoding) usado em
BERT e GPT (5B_4). A ideia de decompor palavras em pedacos menores
e a mesma -- BPE so formaliza o processo de segmentacao.

In [ ]:

# Implementação simplificada de FastText
class SimpleFactText:
    def __init__(self, embedding_dim=5, min_n=3, max_n=6):
        self.embedding_dim = embedding_dim
        self.min_n = min_n
        self.max_n = max_n
        self.subword_embeddings = {}
        self.word_index = {}
    
    def get_subwords(self, word):
        # Adicionar delimitadores
        word = f"#{word}#"
        subwords = set()
        
        # Gerar n-gramas de caracteres
        for n in range(self.min_n, self.max_n + 1):
            for i in range(len(word) - n + 1):
                subwords.add(word[i:i+n])
        
        # Adicionar palavra inteira
        subwords.add(word[1:-1])
        
        return list(subwords)
    
    def fit(self, texts):
        vocab = set()
        for text in texts:
            words = text.lower().split()
            vocab.update(words)
        
        # Inicializar embeddings para subwords
        all_subwords = set()
        word_subwords = {}
        
        for word in vocab:
            subwords = self.get_subwords(word)
            word_subwords[word] = subwords
            all_subwords.update(subwords)
        
        # Atribuir embeddings aleatórios
        for subword in all_subwords:
            self.subword_embeddings[subword] = np.random.randn(self.embedding_dim) * 0.01
        
        self.word_index = {word: idx for idx, word in enumerate(vocab)}
    
    def get_embedding(self, word):
        subwords = self.get_subwords(word)
        # Média dos embeddings dos subwords
        embeddings = [self.subword_embeddings[sw] for sw in subwords 
                     if sw in self.subword_embeddings]
        if embeddings:
            return np.mean(embeddings, axis=0)
        return np.zeros(self.embedding_dim)

# Teste
texts = ['gato gatos gatar', 'cachorro cachorros']
ft = SimpleFactText(embedding_dim=4)
ft.fit(texts)

print("SubwordEmbeddings FastText:")
words_test = ['gato', 'gatos', 'gatar']
for word in words_test:
    subwords = ft.get_subwords(word)
    embedding = ft.get_embedding(word)
    print(f"\n{word}:")
    print(f"  Subwords: {subwords[:5]}...")
    print(f"  Embedding: {embedding}")


## 5. Visualizacao e Avaliacao de Embeddings

### Por que em ML visualizar embeddings e util

Embeddings vivem em 300 dimensoes (tipico). Para ENTENDER o que aprenderam,
projetamos em 2D com PCA ou t-SNE. Clusters no 2D indicam agrupamentos semanticos.

### O que observar sobre PCA vs t-SNE para Embeddings

- **PCA:** preserva distancias globais, mais rapido, deterministico
- **t-SNE:** preserva vizinhancas locais, mais bonito, estocastico
- **UMAP:** equilibrio entre PCA e t-SNE, rapido

Para embeddings, t-SNE ou UMAP sao preferidos porque relacoes locais
(sinonimos proximos) sao mais importantes que distancias globais.

In [ ]:
# Visualizacao: embeddings projetados com PCA manual
np.random.seed(42)

# Simular embeddings de 20 palavras em 10 dimensoes
# Organizados em 4 clusters semanticos
n_words = 20
d = 10

# Clusters: animais, cores, paises, profissoes
clusters = {
    'animais': ['gato', 'cachorro', 'leao', 'tigre', 'urso'],
    'cores': ['vermelho', 'azul', 'verde', 'amarelo', 'roxo'],
    'paises': ['brasil', 'argentina', 'chile', 'peru', 'colombia'],
    'profissoes': ['medico', 'engenheiro', 'advogado', 'professor', 'dentista']
}

# Gerar embeddings com estrutura de cluster
embeddings = {}
for i, (cluster_name, words) in enumerate(clusters.items()):
    center = np.random.randn(d) * 2
    center[i*2:i*2+2] += 3  # Separar clusters em dimensoes especificas
    for word in words:
        embeddings[word] = center + np.random.randn(d) * 0.5

# PCA manual (sem sklearn)
all_words = list(embeddings.keys())
X = np.array([embeddings[w] for w in all_words])
X_centered = X - X.mean(axis=0)

# SVD para PCA
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
X_2d = X_centered @ Vt[:2].T  # Projetar nas 2 primeiras componentes

# Variancia explicada
var_explained = S[:2]**2 / (S**2).sum()

# Visualizar
fig, ax = plt.subplots(figsize=(10, 8))
colors_map = {'animais': 'brown', 'cores': 'red', 'paises': 'blue', 'profissoes': 'green'}

idx = 0
for cluster_name, words in clusters.items():
    for word in words:
        x, y = X_2d[idx]
        ax.scatter(x, y, c=colors_map[cluster_name], s=100, edgecolors='black',
                  linewidth=0.5, zorder=5)
        ax.annotate(word, (x, y), fontsize=9, fontweight='bold',
                    xytext=(5, 5), textcoords='offset points')
        idx += 1

# Legend
for cluster_name, color in colors_map.items():
    ax.scatter([], [], c=color, s=100, label=cluster_name, edgecolors='black')
ax.legend(fontsize=10, loc='upper right')

ax.set_title(f'Embeddings Projetados com PCA\n(var explicada: PC1={var_explained[0]:.1%}, PC2={var_explained[1]:.1%})',
             fontsize=12, fontweight='bold')
ax.set_xlabel('PC1', fontsize=10)
ax.set_ylabel('PC2', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/embedding_clusters.png', dpi=100, bbox_inches='tight')
plt.show()

print('Palavras semanticamente similares formam CLUSTERS no espaco de embeddings')
print('PCA projeta de 10D para 2D preservando a maior variancia')

## 6. Limitacoes de Embeddings Estaticos

### O que observar sobre Polissemia

"Banco" pode significar:
- Instituicao financeira: "fui ao banco sacar dinheiro"
- Assento: "sentei no banco da praca"

Embeddings estaticos (Word2Vec, GloVe, FastText) atribuem UM UNICO vetor
para "banco". Esse vetor e uma MEDIA dos significados -- ruim para ambos!

### Por que em ML embeddings contextuais resolvem polissemia

BERT e GPT geram embeddings DIFERENTES para a mesma palavra dependendo
do contexto:
- "fui ao [banco] sacar" -> embedding de instituicao financeira
- "sentei no [banco] da praca" -> embedding de assento

Isso e possivel porque Transformers veem a frase INTEIRA, nao so a palavra.

### O que concluir sobre quando usar embeddings estaticos vs contextuais

| Cenario | Recomendacao |
|---------|-------------|
| Classificacao simples | Estaticos (rapido, suficiente) |
| NER, QA, NLI | Contextuais (precisam de contexto) |
| Recursos limitados | Estaticos (leve, CPU only) |
| Accuracy maxima | Contextuais (BERT, GPT) |
| Muitos idiomas | FastText (disponivel em 157 idiomas) |

### Conexao com outros notebooks sobre Contexto e Sequencias

A limitacao de embeddings estaticos motiva RNNs/LSTMs (5B_3) e Transformers
(5B_4) -- modelos que processam SEQUENCIAS e geram representacoes contextuais.

In [ ]:

# Exemplo de polissemia
static_embeddings = {
    'banco': np.array([0.5, 0.3, 0.2, 0.1]),  # Uma representação só
}

# Representações contextuais (simuladas)
contextual_bank_financial = np.array([0.6, 0.2, 0.1, 0.0])
contextual_bank_river = np.array([0.2, 0.5, 0.4, 0.3])

print("Polissemia - 'banco':")
print(f"  Estático: {static_embeddings['banco']}")
print(f"  Contexto (financeiro): {contextual_bank_financial}")
print(f"  Contexto (rio): {contextual_bank_river}")

sim_financial = cosine_similarity(
    static_embeddings['banco'], 
    contextual_bank_financial
)
sim_river = cosine_similarity(
    static_embeddings['banco'], 
    contextual_bank_river
)

print(f"\nSimilaridade estático vs contextos:")
print(f"  Financeiro: {sim_financial:.3f}")
print(f"  Rio: {sim_river:.3f}")


## 7. Exercicios Praticos

### Exercicio 1: Similaridade Cosseno

**Tarefa:** Implemente similaridade cosseno e calcule a similaridade entre
pares de palavras usando embeddings simulados.

In [ ]:
# PRATICA - Exercicio 1: Similaridade Cosseno
np.random.seed(42)

# Embeddings simulados (em ViT real seriam 300D, aqui usamos 5D)
embeddings = {
    'gato': np.array([0.8, 0.2, -0.1, 0.5, 0.3]),
    'cachorro': np.array([0.7, 0.3, -0.2, 0.4, 0.35]),
    'carro': np.array([-0.5, 0.8, 0.6, -0.3, 0.1]),
    'moto': np.array([-0.4, 0.7, 0.5, -0.2, 0.15]),
    'feliz': np.array([0.1, -0.3, 0.2, 0.9, 0.7]),
}

def cosine_sim(a, b):
    # TAREFA DO ALUNO: Implementar similaridade cosseno
    # cos(a, b) = dot(a, b) / (norm(a) * norm(b))
    return None

# TAREFA DO ALUNO: Calcular similaridade para todos os pares
pairs = [('gato', 'cachorro'), ('gato', 'carro'), ('carro', 'moto'),
         ('gato', 'feliz'), ('cachorro', 'moto')]

for w1, w2 in pairs:
    sim = cosine_sim(embeddings[w1], embeddings[w2])
    if sim is not None:
        print(f'  sim("{w1}", "{w2}") = {sim:.3f}')
    else:
        print('Complete o TAREFA DO ALUNO acima!')
        break

In [ ]:
# SOLUCAO - Exercicio 1: Similaridade Cosseno
np.random.seed(42)

embeddings = {
    'gato': np.array([0.8, 0.2, -0.1, 0.5, 0.3]),
    'cachorro': np.array([0.7, 0.3, -0.2, 0.4, 0.35]),
    'carro': np.array([-0.5, 0.8, 0.6, -0.3, 0.1]),
    'moto': np.array([-0.4, 0.7, 0.5, -0.2, 0.15]),
    'feliz': np.array([0.1, -0.3, 0.2, 0.9, 0.7]),
}

def cosine_sim(a, b):
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return dot / (norm + 1e-8)

print('Similaridade Cosseno - SOLUCAO')
print('=' * 50)
pairs = [('gato', 'cachorro'), ('gato', 'carro'), ('carro', 'moto'),
         ('gato', 'feliz'), ('cachorro', 'moto')]

for w1, w2 in pairs:
    sim = cosine_sim(embeddings[w1], embeddings[w2])
    print(f'  sim("{w1}", "{w2}") = {sim:.3f}')

print()
print('Observacoes:')
print('  gato-cachorro: ALTA (mesma categoria semantica)')
print('  gato-carro: BAIXA (categorias diferentes)')
print('  carro-moto: ALTA (mesma categoria)')

### Exercicio 2: Resolver Analogias

**Tarefa:** Implemente um resolvedor de analogias (A:B :: C:?) usando
aritmetica de embeddings e busca por vizinho mais proximo.

In [ ]:
# PRATICA - Exercicio 2: Analogias
np.random.seed(42)

# Embeddings simulados com relacoes embutidas
dim = 8
base = np.random.randn(dim)
gender_dir = np.array([1, -1, 0, 0, 0, 0, 0, 0]) * 0.5
royalty_dir = np.array([0, 0, 1, -1, 0, 0, 0, 0]) * 0.5

emb = {
    'rei': base + royalty_dir + gender_dir,
    'rainha': base + royalty_dir - gender_dir,
    'homem': base + gender_dir,
    'mulher': base - gender_dir,
    'principe': base + royalty_dir * 0.5 + gender_dir,
    'princesa': base + royalty_dir * 0.5 - gender_dir,
}

def solve_analogy(a, b, c, embeddings):
    # TAREFA DO ALUNO: Calcular d = b - a + c
    # TAREFA DO ALUNO: Encontrar palavra mais proxima de d (excluindo a, b, c)
    # TAREFA DO ALUNO: Retornar a palavra e a similaridade
    return None, None

# Testar: rei:rainha :: homem:?
word, sim = solve_analogy('rei', 'rainha', 'homem', emb)
if word is not None:
    print(f'rei:rainha :: homem:{word} (sim={sim:.3f})')
else:
    print('Complete os TAREFA DO ALUNOs acima!')

In [ ]:
# SOLUCAO - Exercicio 2: Analogias
np.random.seed(42)

dim = 8
base = np.random.randn(dim)
gender_dir = np.array([1, -1, 0, 0, 0, 0, 0, 0]) * 0.5
royalty_dir = np.array([0, 0, 1, -1, 0, 0, 0, 0]) * 0.5

emb = {
    'rei': base + royalty_dir + gender_dir,
    'rainha': base + royalty_dir - gender_dir,
    'homem': base + gender_dir,
    'mulher': base - gender_dir,
    'principe': base + royalty_dir * 0.5 + gender_dir,
    'princesa': base + royalty_dir * 0.5 - gender_dir,
}

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def solve_analogy(a, b, c, embeddings):
    # d = b - a + c (a relacao de a->b aplicada a c)
    target = embeddings[b] - embeddings[a] + embeddings[c]

    # Buscar mais proximo
    best_word = None
    best_sim = -1
    exclude = {a, b, c}
    for word, vec in embeddings.items():
        if word in exclude:
            continue
        sim = cosine_sim(target, vec)
        if sim > best_sim:
            best_sim = sim
            best_word = word
    return best_word, best_sim

print('Resolver Analogias - SOLUCAO')
print('=' * 50)
analogies = [
    ('rei', 'rainha', 'homem', 'mulher'),
    ('rei', 'rainha', 'principe', 'princesa'),
    ('homem', 'mulher', 'principe', 'princesa'),
]

for a, b, c, expected in analogies:
    result, sim = solve_analogy(a, b, c, emb)
    correct = 'OK' if result == expected else 'ERRO'
    print(f'  [{correct}] {a}:{b} :: {c}:{result} (esperado: {expected}, sim={sim:.3f})')

print()
print('Analogias funcionam porque direcoes no espaco codificam relacoes')
print('genero, realeza, etc. sao VETORES que podem ser somados/subtraidos')

### Exercicio 3: Embedding de Subwords (FastText)

**Tarefa:** Implemente a ideia de FastText: dado um vocabulario de subwords,
calcule o embedding de uma palavra desconhecida como media dos embeddings
de seus subwords.

In [ ]:
# PRATICA - Exercicio 3: FastText Subwords
np.random.seed(42)
dim = 4

# Subword embeddings conhecidos
subword_emb = {
    'corr': np.array([0.5, 0.2, -0.1, 0.3]),
    'endo': np.array([0.1, 0.4, 0.2, -0.1]),
    'endo>': np.array([0.15, 0.35, 0.25, -0.05]),
    '<corr': np.array([0.55, 0.15, -0.15, 0.35]),
    'edor': np.array([0.3, 0.1, 0.5, 0.2]),
    'edor>': np.array([0.35, 0.05, 0.55, 0.25]),
    'idor': np.array([0.25, 0.15, 0.45, 0.15]),
    'ida': np.array([0.2, 0.5, 0.1, 0.0]),
    'rida': np.array([0.3, 0.45, 0.05, 0.1]),
}

def get_subwords(word, n=4):
    # TAREFA DO ALUNO: Gerar n-gramas de caracteres
    # Ex: "correndo" com n=4 -> ["<cor", "corr", "orre", "rren", "rend", "endo", "ndo>"]
    padded = '<' + word + '>'
    subwords = []
    # TAREFA DO ALUNO: gerar subwords
    return subwords

def fasttext_embed(word, subword_embeddings, n=4):
    # TAREFA DO ALUNO: Pegar subwords da palavra
    # TAREFA DO ALUNO: Media dos embeddings dos subwords conhecidos
    return None

# Testar com palavra OOV
result = fasttext_embed('correndo', subword_emb)
if result is not None:
    print(f'Embedding de "correndo": {result}')
else:
    print('Complete os TAREFA DO ALUNOs acima!')

In [ ]:
# SOLUCAO - Exercicio 3: FastText Subwords
np.random.seed(42)
dim = 4

subword_emb = {
    'corr': np.array([0.5, 0.2, -0.1, 0.3]),
    'endo': np.array([0.1, 0.4, 0.2, -0.1]),
    'endo>': np.array([0.15, 0.35, 0.25, -0.05]),
    '<corr': np.array([0.55, 0.15, -0.15, 0.35]),
    'edor': np.array([0.3, 0.1, 0.5, 0.2]),
    'edor>': np.array([0.35, 0.05, 0.55, 0.25]),
    'idor': np.array([0.25, 0.15, 0.45, 0.15]),
    'ida': np.array([0.2, 0.5, 0.1, 0.0]),
    'rida': np.array([0.3, 0.45, 0.05, 0.1]),
}

def get_subwords(word, n=4):
    padded = '<' + word + '>'
    subwords = []
    for i in range(len(padded) - n + 1):
        subwords.append(padded[i:i+n])
    return subwords

def fasttext_embed(word, subword_embeddings, n=4):
    subs = get_subwords(word, n)
    found = []
    for s in subs:
        if s in subword_embeddings:
            found.append(subword_embeddings[s])
    if found:
        return np.mean(found, axis=0)
    return np.zeros(dim)

# Testar
words_test = ['correndo', 'corredor', 'corrida']
print('FastText Subwords - SOLUCAO')
print('=' * 50)
for word in words_test:
    subs = get_subwords(word)
    known = [s for s in subs if s in subword_emb]
    embed = fasttext_embed(word, subword_emb)
    print(f'  "{word}":')
    print(f'    Subwords: {subs}')
    print(f'    Conhecidos: {known}')
    print(f'    Embedding: {embed.round(3)}')
    print()

# Similaridade entre formas da mesma raiz
e1 = fasttext_embed('correndo', subword_emb)
e2 = fasttext_embed('corredor', subword_emb)
sim = np.dot(e1, e2) / (np.linalg.norm(e1) * np.linalg.norm(e2) + 1e-8)
print(f'Similaridade "correndo" vs "corredor": {sim:.3f}')
print('Palavras com mesma raiz tem embeddings SIMILARES mesmo sem ter visto no treino!')

### O que observar sobre Qualidade de Embeddings

A qualidade de um embedding depende criticamente do corpus de treino:
- Corpus de noticias: boas relacoes factuais, fraco em gírias
- Corpus de Twitter: bom para sentimento coloquial, ruidoso
- Corpus medico: captura terminologia clinica, mas vocabulario restrito
- Corpus multilingual: permite transferencia cross-lingual

**Tamanho importa:** Word2Vec treinado em 100B tokens >> treinado em 1M tokens.
Embeddings pre-treinados (GloVe 840B, FastText crawl) sao preferidos na pratica.

### O que concluir sobre Dimensionalidade de Embeddings

A dimensao ideal depende da tarefa e do tamanho do vocabulario:
- 50D: suficiente para tarefas simples, rapido de treinar
- 100-200D: melhor custo-beneficio para a maioria das aplicacoes
- 300D: padrao em pesquisa (GloVe, Word2Vec original)
- 1024D+: retornos decrescentes, risco de overfitting

Mais dimensoes = mais capacidade expressiva, mas tambem mais dados necessarios
para preencher o espaco de forma util (maldicao da dimensionalidade).

### Conexao com outros notebooks sobre Representacao de Features

Embeddings sao uma forma de **aprendizado de representacao** (1_4 feature engineering).
Assim como PCA (3_2) reduz dimensionalidade preservando variancia, embeddings
reduzem vocabulario para vetores densos preservando relacoes semanticas.

### O que observar sobre Bias em Word Embeddings

Embeddings herdam e amplificam vieses do corpus:
- "medico" mais proximo de "homem", "enfermeira" mais proximo de "mulher"
- Associacoes raciais e etnicas codificadas nos vetores
- Técnicas de debiasing existem mas sao imperfeitas

Isso e critico em aplicacoes de producao: sistemas de recrutamento, credito,
e justica podem perpetuar discriminacao via embeddings enviesados.

### O que concluir sobre Evolucao de Representacoes Textuais

A evolucao e clara: one-hot (esparso) -> BoW/TF-IDF (esparso com frequencia) ->
Word2Vec/GloVe (denso estatico) -> ELMo (denso contextual) -> BERT/GPT (denso
contextual com attention). Cada passo resolve limitacoes do anterior.

### Conexao com outros notebooks sobre Transfer Learning em NLP

Embeddings pre-treinados foram o PRIMEIRO transfer learning em NLP (5B_3 e 5B_4).
Antes de BERT, o pipeline padrao era: carregar GloVe -> fine-tune na tarefa.
Hoje, embeddings contextuais (BERT) substituiram embeddings estaticos na maioria
das tarefas, mas GloVe/FastText ainda sao usados em sistemas de baixa latencia.

### O que concluir sobre Avaliacao de Embeddings

Existem dois tipos de avaliacao:
- **Intrinseca:** analogias (rei:rainha::homem:mulher), similaridade (WordSim-353)
- **Extrinseca:** performance na tarefa final (classificacao, NER, traducao)

Embeddings com boa avaliacao intrinseca NEM SEMPRE sao os melhores para a tarefa
final. A avaliacao extrinseca e o que importa na pratica.

### Conexao com outros notebooks sobre Reducao de Dimensionalidade

A visualizacao de embeddings com PCA conecta diretamente com 3_2 (PCA).
t-SNE e UMAP sao alternativas nao-lineares que preservam estrutura local
melhor que PCA, mas sao mais lentas e nao-deterministicas.

## 8. Erros Comuns e Armadilhas

### Erro 1: Usar Embeddings Pre-treinados sem Fine-tuning

**O que acontece:** Embeddings treinados em Wikipedia nao funcionam em textos medicos.

**Por que em ML:** Embeddings capturam a distribuicao do CORPUS DE TREINO.
"Celula" em Wikipedia = biologia. "Celula" em telecom = aparelho.

**Solucao:** Fine-tunar embeddings no seu dominio, ou usar embeddings contextuais.

### Erro 2: Dimensionalidade Muito Alta ou Baixa

**O que acontece:** 50D = pouca expressividade. 1000D = overfitting.

**Por que em ML:** Embeddings de 300D sao o sweet spot para a maioria das tarefas.
50D perde nuances semanticas. 1000D overfita para corpus pequenos.

**Solucao:** Comecar com 300D. Testar 100D e 500D. Escolher por validacao.

### Erro 3: Ignorar Bias nos Embeddings

**O que acontece:** Embeddings aprendem estereotipos do corpus de treino.
"Programador" fica proximo de "homem", "enfermeira" de "mulher".

**Por que em ML:** Se o corpus tem bias, os embeddings reproduzem esse bias.

**Solucao:** Debiasing tecnico (projecao ortogonal), ou usar dados balanceados.

### Erro 4: Comparar Embeddings de Modelos Diferentes

**O que acontece:** Similaridade entre Word2Vec e GloVe nao faz sentido.

**Por que em ML:** Cada modelo define seu proprio espaco. Dimensoes nao tem
significado absoluto -- so RELATIVO dentro do mesmo modelo.

**Solucao:** Sempre comparar embeddings do MESMO modelo e treinamento.

### Erro 5: Confundir Embeddings Estaticos com Contextuais

**O que acontece:** Usar Word2Vec onde BERT seria necessario (ou vice-versa).

**Por que em ML:** Embeddings estaticos sao RAPIDOS mas FIXOS por palavra.
Contextuais sao LENTOS mas ADAPTAM ao contexto.

### O que observar sobre Embeddings Pre-treinados Disponiveis

Embeddings pre-treinados mais populares:
- **Word2Vec:** GoogleNews (3M palavras, 300D, ingles)
- **GloVe:** Varios tamanhos (50D-300D, varios corpus, ingles)
- **FastText:** 157 idiomas (300D), incluindo portugues
- **NILC:** Embeddings em portugues brasileiro (varias dimensoes)

Para portugues, FastText pre-treinado e a melhor opcao para embeddings estaticos.

### O que concluir sobre o Papel Atual de Embeddings Estaticos

Embeddings estaticos nao sao "obsoletos" -- ainda sao usados em:
1. Inicializacao de modelos maiores (embeddings como warm start)
2. Sistemas de busca e recomendacao (rapido, eficiente)
3. Analise exploratoria (visualizar relacoes semanticas)
4. Aplicacoes com restricoes de compute (IoT, mobile)

### Conexao com outros notebooks sobre o Pipeline NLP Moderno

Embeddings sao a CAMADA 0 de qualquer modelo NLP moderno:
- BERT: embedding layer + positional encoding (5B_4)
- GPT: token embeddings + position embeddings (5B_5)
- Mesmo LLMs com trilhoes de parametros comecam com uma camada de embedding

### O que observar sobre Avaliacao de Embeddings

Metricas para avaliar qualidade de embeddings:
1. **Analogias:** king - man + woman = queen (accuracy)
2. **Similaridade:** correlacao com scores humanos (Spearman)
3. **Downstream:** accuracy em tarefa final (classificacao, NER)
4. **Intriseca vs Extrinseca:** analogias medem o embedding, downstream mede a utilidade

### O que concluir sobre a Importancia de Embeddings na Historia de NLP

Embeddings foram o "momento ImageNet" de NLP. Antes de Word2Vec (2013),
NLP era dominado por features manuais. Depois, representacoes aprendidas
se tornaram o padrao. Isso abriu caminho para BERT (2018) e GPT (2020),
que sao essencialmente embeddings contextuais massivos.

### Conexao com outros notebooks sobre Representacoes Aprendidas

A ideia de "aprender representacoes" conecta com autoencoders (5C_1),
contrastive learning (5A_5), e transfer learning (5A_2). Em todos os casos,
a ideia central e: deixar o MODELO descobrir as melhores features,
em vez de engenheirar manualmente.

## Resumo e Proximos Passos

### Hierarquia de Conceitos

```
Word Embeddings
|
|-- Motivacao
|   |-- Limitacoes de BoW/TF-IDF (sparse, sem semantica)
|   |-- Hipotese distribucional (contexto = significado)
|   +-- Embeddings densos (representacao compacta)
|
|-- Modelos
|   |-- Word2Vec (Skip-gram, CBOW, negative sampling)
|   |-- GloVe (co-ocorrencia global, fatoracao de matriz)
|   +-- FastText (subwords, OOV handling)
|
|-- Propriedades
|   |-- Aritmetica de embeddings (analogias)
|   |-- Clusters semanticos
|   +-- Bias (refletem o corpus de treino)
|
+-- Limitacoes
    |-- Polissemia (uma palavra = um vetor)
    |-- Contexto (nao muda com a frase)
    +-- -> Motivam embeddings contextuais (BERT, GPT)
```

### Conexoes entre Notebooks

| Conceito | Conecta com | Relacao |
|----------|-------------|---------|
| Embeddings densos | 3_2 (PCA/SVD) | Reducao de dimensionalidade |
| Word2Vec | 5A_5 (self-supervised) | Aprender sem labels |
| FastText subwords | 5B_4 (BPE) | Tokenizacao subword |
| Polissemia | 5B_4 (BERT) | Embeddings contextuais |
| Bias | 4_1 (regularizacao) | Bias no modelo |
| Similarity | 1_2 (metricas) | Cosseno vs Euclidiana |

### Checklist de Competencias

- [ ] Entendo limitacoes de BoW/TF-IDF (sparse, sem semantica)
- [ ] Sei como Word2Vec aprende embeddings (skip-gram, CBOW)
- [ ] Entendo GloVe e co-ocorrencia global
- [ ] Sei como FastText resolve OOV com subwords
- [ ] Consigo calcular similaridade cosseno e resolver analogias
- [ ] Entendo a limitacao de polissemia em embeddings estaticos
- [ ] Sei quando usar embeddings estaticos vs contextuais

### Proximos Passos

- **5B_3:** RNNs e LSTMs (processar sequencias de embeddings)
- **5B_4:** Transformers e BERT (embeddings contextuais)
- **Pratica:** Usar FastText pre-treinado para classificacao em portugues